# UHG Intrusion Detection System v4.9

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zbovaird/UHG-Library/blob/main/tests/UHG_IDS_4_9.ipynb)

**Universal Hyperbolic Geometry (UHG) Graph Neural Network for Network Intrusion Detection**

v4.9 improvements over v4.8.2:
- **FAISS GPU** KNN with PyNNDescent fallback (graph build ~1-3s vs ~27s)
- **Focal Loss** replaces weighted CrossEntropy for extreme class imbalance
- **Stratified sampling** with minimum-per-class floor (rare classes preserved)
- **Undirected KNN graph** (bidirectional edges for better message passing)
- **k=5 neighbors** (up from k=2 for denser graph structure)
- **Larger model** (128 hidden, 3 layers vs 64 hidden, 2 layers)
- **PyArrow CSV engine** (3-5x faster data loading)
- **Consolidated preprocessing** (fewer pandas passes)
- **Test eval only at end** (skip redundant forward passes during training)

Dataset: CIC-IDS2017 (15 classes, ~2.8M samples, 77 features)

In [ ]:
# Install dependencies
!pip -q install --upgrade pip
import torch
pt = torch.__version__.split('+')[0]
cuda = torch.version.cuda
if torch.cuda.is_available() and cuda:
    idx = f"https://data.pyg.org/whl/torch-{pt}+cu{cuda.replace('.','')}.html"
else:
    idx = f"https://data.pyg.org/whl/torch-{pt}+cpu.html"

!pip -q install torch_scatter torch_sparse torch_cluster torch_spline_conv -f {idx}
!pip -q install torch_geometric scikit-learn scipy pandas tqdm pyarrow

# Install FAISS (GPU if available, CPU fallback)
try:
    if torch.cuda.is_available():
        !pip -q install faiss-gpu
    else:
        !pip -q install faiss-cpu
except Exception:
    pass

# PyNNDescent as fallback
!pip -q install pynndescent

In [ ]:
"""
Intrusion Detection using Universal Hyperbolic Geometry (UHG) v4.9

v4.9 CHANGES (from v4.8.2):
  Speed:
  - FAISS GPU KNN (fallback: PyNNDescent) -> ~10-25x faster graph build
  - PyArrow CSV engine -> 3-5x faster data load
  - Consolidated preprocessing (fewer fillna passes)
  - Test evaluation only at end of training

  Accuracy:
  - k=5 neighbors (was k=2) -> denser graph, better message passing
  - Undirected edges (was directed) -> bidirectional information flow
  - Focal Loss with gamma=2 (was weighted CE) -> better minority class focus
  - Stratified sampling with min 50 per class (was random) -> rare classes preserved
  - hidden=128, layers=3 (was 64, 2) -> more model capacity
  - Stratified train/val/test split -> every class in every split
  - UHG norm regularization loss term -> constraint preservation

  v4.8.2 baseline: 94.46% acc, 0.47 macro-F1, 93s total
  v4.9 target:     94-96% acc, 0.65-0.80 macro-F1, 40-60s total
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from torch_geometric.data import Data
from torch_scatter import scatter_add
from torch_geometric.nn.conv import MessagePassing
from typing import Tuple, Optional
import os
import sys
import time
import json
import traceback
import platform
from datetime import datetime

# KNN backends
FAISS_AVAILABLE = False
try:
    import faiss
    FAISS_AVAILABLE = True
except ImportError:
    pass

PYNNDESCENT_AVAILABLE = False
try:
    from pynndescent import NNDescent
    PYNNDESCENT_AVAILABLE = True
except ImportError:
    pass

# Optional: Drive mount (only in Colab)
try:
    from google.colab import drive
    print("Mounting Google Drive...")
    drive.mount('/content/drive')
except Exception:
    pass

# ==============================================================================
# CONFIGURATION
# ==============================================================================
WEIGHTING_STRATEGY = "sqrt"     # Options: "sqrt", "log", "capped"
FOCAL_GAMMA = 2.0               # Focal loss gamma (0 = plain CE)
K_NEIGHBORS = 5                 # KNN neighbors (was 2 in v4.8.2)
HIDDEN_CHANNELS = 128           # Hidden dim (was 64 in v4.8.2)
NUM_LAYERS = 3                  # GNN depth (was 2 in v4.8.2)
DROPOUT = 0.3                   # Dropout rate
SAMPLE_FRAC = 0.20              # Fraction of data to sample
MIN_SAMPLES_PER_CLASS = 50      # Minimum samples per class after sampling
PCA_COMPONENTS = 20             # PCA dims for KNN
LR = 0.005                      # Initial learning rate
WEIGHT_DECAY = 1e-5             # Adam weight decay
PATIENCE = 12                   # Early stopping patience
MAX_EPOCHS = 200                # Maximum training epochs
UHG_REG_LAMBDA = 0.01          # UHG constraint regularization weight
UNDIRECTED = True               # Make KNN graph undirected
# ==============================================================================

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("=" * 80)
print("  UHG INTRUSION DETECTION SYSTEM v4.9")
print("=" * 80)

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory = torch.cuda.get_device_properties(0).total_mem / 1024**3
    print(f"GPU: {gpu_name} ({gpu_memory:.1f} GB) | CUDA {torch.version.cuda}")
else:
    print("No GPU available - using CPU")

print(f"KNN backend: {'FAISS GPU' if (FAISS_AVAILABLE and torch.cuda.is_available()) else 'FAISS CPU' if FAISS_AVAILABLE else 'PyNNDescent' if PYNNDESCENT_AVAILABLE else 'sklearn (slow!)'}")
print(f"Config: k={K_NEIGHBORS}, hidden={HIDDEN_CHANNELS}, layers={NUM_LAYERS}, "
      f"focal_gamma={FOCAL_GAMMA}, strategy={WEIGHTING_STRATEGY}, "
      f"undirected={UNDIRECTED}, min_per_class={MIN_SAMPLES_PER_CLASS}")
print("=" * 80 + "\n")

# ==============================================================================
# File Configuration
# ==============================================================================
FILE_PATH = '/content/drive/MyDrive/CIC_data.csv'
MODEL_SAVE_PATH = '/content/drive/MyDrive/uhg_ids_model_best_v49.pth'
RESULTS_PATH = '/content/drive/MyDrive/uhg_ids_results/'
os.makedirs(RESULTS_PATH, exist_ok=True)


# ==============================================================================
# UHG Geometry Helpers
# ==============================================================================

def minkowski_inner_product(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Minkowski inner product: <x,y>_M = sum(x_i*y_i) - x_t*y_t"""
    spatial = (x[..., :-1] * y[..., :-1]).sum(dim=-1)
    time = x[..., -1] * y[..., -1]
    return spatial - time


def projective_normalize(x: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Normalize projective coordinates: x^2 + y^2 - z^2 = -1"""
    spatial_norm_sq = (x[..., :-1] ** 2).sum(dim=-1, keepdim=True)
    z = torch.sqrt(torch.clamp(spatial_norm_sq + 1.0, min=eps))
    return torch.cat([x[..., :-1], z], dim=-1)


def uhg_quadrance_vectorized(x: torch.Tensor, y: torch.Tensor, eps: float = 1e-8) -> torch.Tensor:
    """Compute vectorized UHG quadrance between vectors."""
    numerator = minkowski_inner_product(x, y)
    denom_x = torch.clamp(-minkowski_inner_product(x, x), min=eps)
    denom_y = torch.clamp(-minkowski_inner_product(y, y), min=eps)
    cos_val = numerator / torch.sqrt(denom_x * denom_y)
    cos_val = torch.clamp(cos_val, min=-1.0 + eps, max=1.0 - eps)
    return 1 - cos_val ** 2


def verify_uhg_constraints(x: torch.Tensor, name: str = "embeddings"):
    """Verify Minkowski norm constraints."""
    norm_sq = minkowski_inner_product(x, x)
    violation = torch.abs(norm_sq + 1.0)
    max_viol = violation.max().item()
    mean_viol = violation.mean().item()
    print(f"UHG Constraint Check ({name}):")
    print(f"  Max violation: {max_viol:.6f}")
    print(f"  Mean violation: {mean_viol:.6f}")
    if max_viol > 0.01:
        print(f"  WARNING: Constraints violated!")
    else:
        print(f"  Constraints satisfied.")


# ==============================================================================
# Focal Loss
# ==============================================================================

class FocalLoss(nn.Module):
    """Focal Loss for imbalanced classification.

    FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    When gamma=0, this reduces to standard weighted cross-entropy.
    Higher gamma focuses more on hard/misclassified examples.
    """
    def __init__(self, alpha: Optional[torch.Tensor] = None, gamma: float = 2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        ce_loss = F.cross_entropy(inputs, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()


# ==============================================================================
# Data Loading
# ==============================================================================

def compute_class_weights(class_counts: np.ndarray, strategy: str = "sqrt") -> np.ndarray:
    """Compute class weights with dampening."""
    total_samples = class_counts.sum()
    num_classes = len(class_counts)
    raw_weights = total_samples / (num_classes * np.maximum(class_counts, 1))

    if strategy == "sqrt":
        weights = np.sqrt(raw_weights)
    elif strategy == "log":
        weights = np.log1p(raw_weights)
    elif strategy == "capped":
        weights = np.clip(raw_weights, 0.1, 100.0)
    else:
        raise ValueError(f"Unknown weighting strategy: {strategy}")

    return weights


def load_and_preprocess_data(
    file_path: str = FILE_PATH,
    sample_frac: float = SAMPLE_FRAC,
    min_per_class: int = MIN_SAMPLES_PER_CLASS,
    weighting_strategy: str = WEIGHTING_STRATEGY,
) -> Tuple[torch.Tensor, torch.Tensor, dict, torch.Tensor, dict]:
    """
    v4.9: Stratified sampling, PyArrow engine, consolidated preprocessing.

    Returns:
        node_features, labels_tensor, label_mapping, class_weights, timings
    """
    timings = {}

    # --- CSV Read (PyArrow engine for speed) ---
    print(f"Loading data from: {file_path}")
    t0 = time.perf_counter()
    try:
        data = pd.read_csv(file_path, low_memory=False, engine='pyarrow')
    except Exception:
        # Fallback to default engine if pyarrow unavailable
        data = pd.read_csv(file_path, low_memory=False)
    timings['csv_read'] = time.perf_counter() - t0
    print(f"  CSV read: {timings['csv_read']:.2f}s ({len(data):,} rows)")

    # --- Column cleanup ---
    t0 = time.perf_counter()
    data.columns = data.columns.str.strip()
    data['Label'] = data['Label'].str.strip()
    timings['column_cleanup'] = time.perf_counter() - t0

    print(f"\nLabel distribution (full dataset):")
    print(data['Label'].value_counts())

    # --- Stratified sampling with minimum-per-class floor ---
    print(f"\nStratified sampling (frac={sample_frac}, min_per_class={min_per_class})...")
    t0 = time.perf_counter()

    # Step 1: Random sample of the target fraction
    data_sampled = data.sample(frac=sample_frac, random_state=42)

    # Step 2: Ensure minimum representation per class
    sampled_counts = data_sampled['Label'].value_counts()
    extra_frames = []
    for label in data['Label'].unique():
        current_count = sampled_counts.get(label, 0)
        if current_count < min_per_class:
            # Get all rows of this class not already sampled
            pool = data[data['Label'] == label]
            needed = min_per_class - current_count
            if len(pool) >= min_per_class:
                # Sample additional from the full dataset (may include duplicates from sampled)
                extra = pool.sample(n=needed, random_state=42, replace=(len(pool) < needed))
            else:
                # Class has fewer than min_per_class total; oversample with replacement
                extra = pool.sample(n=needed, random_state=42, replace=True)
            extra_frames.append(extra)
            print(f"  Boosted '{label}': {current_count} -> {current_count + len(extra)} samples")

    if extra_frames:
        data_sampled = pd.concat([data_sampled] + extra_frames, ignore_index=True)

    timings['sampling'] = time.perf_counter() - t0
    print(f"  Sampling: {timings['sampling']:.2f}s ({len(data_sampled):,} rows)")

    print(f"\nSampled label distribution:")
    print(data_sampled['Label'].value_counts())

    # --- Separate labels before numeric conversion ---
    labels = data_sampled['Label'].copy()
    feature_cols = [c for c in data_sampled.columns if c != 'Label']

    # --- Convert features to numeric + fill NaN/inf in one pass ---
    t0 = time.perf_counter()
    features_df = data_sampled[feature_cols].apply(pd.to_numeric, errors='coerce')
    # Compute column means for fillna (ignoring inf)
    features_np = features_df.values.astype(np.float64)
    features_np[~np.isfinite(features_np)] = np.nan
    col_means = np.nanmean(features_np, axis=0)
    col_means = np.nan_to_num(col_means, nan=0.0)
    # Fill NaN with column means
    inds = np.where(np.isnan(features_np))
    features_np[inds] = np.take(col_means, inds[1])
    timings['preprocess'] = time.perf_counter() - t0
    print(f"  Preprocessing: {timings['preprocess']:.2f}s")

    # --- Scaling ---
    t0 = time.perf_counter()
    scaler = StandardScaler()
    features_scaled = scaler.fit_transform(features_np)
    timings['scaling'] = time.perf_counter() - t0
    print(f"  Scaling: {timings['scaling']:.2f}s")

    # --- Label encoding ---
    unique_labels = sorted(labels.unique())
    label_mapping = {label: idx for idx, label in enumerate(unique_labels)}
    labels_numeric = labels.map(label_mapping).values

    # --- Convert to tensors ---
    t0 = time.perf_counter()
    node_features = torch.tensor(features_scaled, dtype=torch.float32)
    labels_tensor = torch.tensor(labels_numeric, dtype=torch.long)
    timings['to_tensors'] = time.perf_counter() - t0

    print(f"\nFeature shape: {node_features.shape}")
    print(f"Unique labels: {len(unique_labels)}")

    # --- Class weights ---
    class_counts = np.bincount(labels_numeric, minlength=len(unique_labels))
    print(f"\nClass distribution:")
    for label, idx in sorted(label_mapping.items(), key=lambda x: x[1]):
        count = class_counts[idx]
        pct = (count / len(labels_numeric)) * 100
        print(f"  {label:30s}: {count:7d} ({pct:5.2f}%)")

    class_weights = compute_class_weights(class_counts, strategy=weighting_strategy)
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)

    timings['total'] = sum(timings.values())
    print(f"\nData loading total: {timings['total']:.2f}s")

    return node_features, labels_tensor, label_mapping, class_weights_tensor, timings


# ==============================================================================
# Graph Construction (KNN)
# ==============================================================================

def knn_faiss(features: np.ndarray, k: int) -> np.ndarray:
    """KNN using FAISS (GPU if available, else CPU)."""
    n, d = features.shape
    features = np.ascontiguousarray(features, dtype=np.float32)

    if torch.cuda.is_available():
        res = faiss.StandardGpuResources()
        index = faiss.GpuIndexFlatL2(res, d)
    else:
        index = faiss.IndexFlatL2(d)

    index.add(features)
    # k+1 because first neighbor is self
    distances, indices = index.search(features, k + 1)
    # Remove self-neighbor (column 0)
    return indices[:, 1:]


def knn_pynndescent(features: np.ndarray, k: int) -> np.ndarray:
    """KNN using PyNNDescent (approximate, CPU)."""
    index = NNDescent(
        features,
        n_neighbors=k + 1,
        metric='euclidean',
        n_jobs=-1,
        verbose=False,
    )
    indices, _ = index.neighbor_graph
    return indices[:, 1:]  # skip self


def create_graph_data(
    node_features: torch.Tensor,
    labels: torch.Tensor,
    k: int = K_NEIGHBORS,
    pca_components: int = PCA_COMPONENTS,
    undirected: bool = UNDIRECTED,
) -> Tuple[Data, dict]:
    """v4.9: FAISS GPU KNN, undirected edges, stratified split."""
    timings = {}

    print("\nCreating graph structure...")
    features_np = node_features.cpu().numpy()

    # --- PCA for faster KNN ---
    if features_np.shape[1] > pca_components:
        print(f"  PCA: {features_np.shape[1]} -> {pca_components} dims...")
        t0 = time.perf_counter()
        pca = PCA(n_components=pca_components)
        features_reduced = pca.fit_transform(features_np)
        timings['pca'] = time.perf_counter() - t0
        expl = pca.explained_variance_ratio_.sum()
        print(f"  PCA explained variance: {expl:.4f} ({expl*100:.1f}%) | {timings['pca']:.2f}s")
        features_for_knn = features_reduced
    else:
        features_for_knn = features_np
        timings['pca'] = 0.0

    # --- KNN (FAISS GPU > FAISS CPU > PyNNDescent) ---
    print(f"  Computing KNN (k={k})...")
    t0 = time.perf_counter()

    if FAISS_AVAILABLE:
        backend = 'FAISS GPU' if torch.cuda.is_available() else 'FAISS CPU'
        print(f"  Backend: {backend}")
        indices = knn_faiss(features_for_knn.astype(np.float32), k)
    elif PYNNDESCENT_AVAILABLE:
        print(f"  Backend: PyNNDescent (approximate)")
        indices = knn_pynndescent(features_for_knn.astype(np.float32), k)
    else:
        raise RuntimeError("No KNN backend available. Install faiss-gpu or pynndescent.")

    timings['knn'] = time.perf_counter() - t0
    print(f"  KNN done: {timings['knn']:.2f}s")

    # --- Build edge index ---
    t0 = time.perf_counter()
    num_nodes = features_for_knn.shape[0]
    row = np.repeat(np.arange(num_nodes), k)
    col = indices.flatten()
    edge_index = torch.from_numpy(np.vstack([row, col])).long()

    # Make undirected by adding reverse edges
    if undirected:
        edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)
        # Remove duplicates
        edge_index = torch.unique(edge_index, dim=1)

    edge_index = edge_index.to(device)
    timings['edge_build'] = time.perf_counter() - t0
    print(f"  Edges: {edge_index.shape[1]:,} ({'undirected' if undirected else 'directed'})")

    # --- UHG projection ---
    t0 = time.perf_counter()
    node_features_uhg = torch.cat([
        node_features.to(device),
        torch.ones(node_features.size(0), 1, device=device),
    ], dim=1)
    node_features_uhg = projective_normalize(node_features_uhg)
    timings['uhg_projection'] = time.perf_counter() - t0
    print(f"  UHG projection: {timings['uhg_projection']:.2f}s")
    print(f"  Feature shape (with homogeneous coord): {node_features_uhg.shape}")

    verify_uhg_constraints(node_features_uhg, name="initial features")

    # --- Stratified train/val/test split ---
    t0 = time.perf_counter()
    labels_np = labels.cpu().numpy()
    all_indices = np.arange(len(labels_np))

    # First split: 70% train, 30% temp
    train_idx, temp_idx = train_test_split(
        all_indices, test_size=0.30, random_state=42, stratify=labels_np
    )
    # Second split: 50/50 of temp -> 15% val, 15% test
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.50, random_state=42, stratify=labels_np[temp_idx]
    )

    total_samples = len(labels_np)
    train_mask = torch.zeros(total_samples, dtype=torch.bool, device=device)
    val_mask = torch.zeros(total_samples, dtype=torch.bool, device=device)
    test_mask = torch.zeros(total_samples, dtype=torch.bool, device=device)
    train_mask[train_idx] = True
    val_mask[val_idx] = True
    test_mask[test_idx] = True
    timings['split'] = time.perf_counter() - t0

    print(f"  Train: {train_mask.sum().item():,}  Val: {val_mask.sum().item():,}  Test: {test_mask.sum().item():,}")

    timings['total'] = sum(timings.values())
    print(f"  Graph construction total: {timings['total']:.2f}s")

    return Data(
        x=node_features_uhg,
        edge_index=edge_index,
        y=labels.to(device),
        train_mask=train_mask,
        val_mask=val_mask,
        test_mask=test_mask,
    ).to(device), timings


# ==============================================================================
# UHG GraphSAGE Message Passing
# ==============================================================================

class UHGMessagePassing(MessagePassing):
    def __init__(self, in_features: int, out_features: int):
        super().__init__(aggr='add')
        self.in_features = in_features
        self.out_features = out_features
        self.weight_msg = nn.Parameter(torch.Tensor(in_features, out_features))
        self.weight_node = nn.Parameter(torch.Tensor(in_features, out_features))
        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.weight_msg)
        nn.init.xavier_uniform_(self.weight_node)

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        features = x[:, :-1]
        z = x[:, -1:]
        transformed_features = features @ self.weight_node
        out = self.propagate(edge_index, x=x, size=(x.size(0), x.size(0)))
        out = out + transformed_features
        out_full = torch.cat([out, z], dim=1)
        out_full = projective_normalize(out_full)
        return out_full

    def message(self, x_i: torch.Tensor, x_j: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        weights = torch.exp(-uhg_quadrance_vectorized(x_i, x_j))
        messages = (x_j[:, :-1]) @ self.weight_msg
        return messages * weights.view(-1, 1)

    def aggregate(self, inputs: torch.Tensor, index: torch.Tensor, ptr=None, dim_size=None) -> torch.Tensor:
        numerator = scatter_add(inputs, index, dim=0, dim_size=dim_size)
        weights_sum = scatter_add(torch.ones_like(inputs), index, dim=0, dim_size=dim_size)
        return numerator / torch.clamp(weights_sum, min=1e-6)


class UHGGraphSAGE(nn.Module):
    def __init__(self, in_channels: int, hidden_channels: int, out_channels: int,
                 num_layers: int, dropout: float = 0.3):
        super().__init__()
        self.layers = nn.ModuleList()
        self.norms = nn.ModuleList()  # Batch norm for spatial features
        self.dropout = nn.Dropout(dropout)
        actual_in = in_channels - 1  # exclude homogeneous coord

        self.layers.append(UHGMessagePassing(actual_in, hidden_channels))
        self.norms.append(nn.BatchNorm1d(hidden_channels))
        for _ in range(num_layers - 2):
            self.layers.append(UHGMessagePassing(hidden_channels, hidden_channels))
            self.norms.append(nn.BatchNorm1d(hidden_channels))
        self.layers.append(UHGMessagePassing(hidden_channels, out_channels))

    def forward(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        h = x
        for i, layer in enumerate(self.layers[:-1]):
            h = layer(h, edge_index)
            spatial = h[:, :-1]
            spatial = self.norms[i](spatial)
            spatial = F.relu(spatial)
            spatial = self.dropout(spatial)
            h = torch.cat([spatial, h[:, -1:]], dim=1)
        h = self.layers[-1](h, edge_index)
        return h[:, :-1]  # logits on spatial part

    def get_embeddings(self, x: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
        """Return intermediate embeddings for constraint checking."""
        h = x
        for i, layer in enumerate(self.layers[:-1]):
            h = layer(h, edge_index)
            spatial = h[:, :-1]
            spatial = self.norms[i](spatial)
            spatial = F.relu(spatial)
            h = torch.cat([spatial, h[:, -1:]], dim=1)
        return h  # Last hidden state (before final layer)


# ==============================================================================
# Training / Evaluation
# ==============================================================================

def train_epoch(model, graph_data, optimizer, criterion, uhg_reg_lambda=UHG_REG_LAMBDA):
    """Train one epoch with optional UHG constraint regularization."""
    model.train()
    optimizer.zero_grad(set_to_none=True)

    out = model(graph_data.x, graph_data.edge_index)
    loss = criterion(out[graph_data.train_mask], graph_data.y[graph_data.train_mask])

    # UHG constraint regularization: penalize Minkowski norm deviations
    if uhg_reg_lambda > 0:
        embeddings = model.get_embeddings(graph_data.x, graph_data.edge_index)
        norm_sq = minkowski_inner_product(embeddings, embeddings)
        uhg_penalty = ((norm_sq + 1.0) ** 2).mean()
        loss = loss + uhg_reg_lambda * uhg_penalty

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()

    return float(loss.item())


@torch.no_grad()
def evaluate(model, graph_data, mask):
    """Evaluate accuracy on a masked subset."""
    model.eval()
    out = model(graph_data.x, graph_data.edge_index)
    pred = out[mask].argmax(dim=1)
    acc = (pred == graph_data.y[mask]).float().mean().item()
    return acc


@torch.no_grad()
def evaluate_detailed(model, graph_data, mask, label_mapping, phase="Test"):
    """Detailed per-class evaluation."""
    model.eval()
    out = model(graph_data.x, graph_data.edge_index)
    pred = out[mask].argmax(dim=1).cpu().numpy()
    true = graph_data.y[mask].cpu().numpy()

    idx_to_label = {v: k for k, v in label_mapping.items()}
    unique_classes = np.unique(np.concatenate([true, pred]))
    target_names = [idx_to_label[i] for i in unique_classes]

    all_classes = set(range(len(label_mapping)))
    present_classes = set(unique_classes)
    missing_classes = all_classes - present_classes

    print(f"\n{'=' * 80}")
    print(f"{phase} Set - Detailed Performance Report")
    print(f"{'=' * 80}")

    if missing_classes:
        print(f"\nWARNING: {len(missing_classes)} classes not present in {phase.lower()} set:")
        for class_idx in sorted(missing_classes):
            print(f"  - {idx_to_label[class_idx]}")

    overall_acc = (pred == true).mean()
    print(f"\nOverall Accuracy: {overall_acc:.4f}")
    print(f"Classes evaluated: {len(unique_classes)}/{len(label_mapping)}")

    print("\nPer-Class Classification Report:")
    print(classification_report(
        true, pred, labels=unique_classes,
        target_names=target_names, zero_division=0, digits=4,
    ))

    cm = confusion_matrix(true, pred)
    print("Per-Class Accuracy:")
    for i, label in enumerate(target_names):
        class_acc = cm[i, i] / cm[i].sum() if cm[i].sum() > 0 else 0.0
        class_samples = cm[i].sum()
        print(f"  {label:30s}: {class_acc:.4f} ({int(class_samples)} samples)")

    f1_macro = f1_score(true, pred, average='macro', zero_division=0)
    f1_weighted = f1_score(true, pred, average='weighted', zero_division=0)
    print(f"\nF1 Score (Macro):    {f1_macro:.4f}")
    print(f"F1 Score (Weighted): {f1_weighted:.4f}")

    return {
        'accuracy': float(overall_acc),
        'f1_macro': float(f1_macro),
        'f1_weighted': float(f1_weighted),
        'confusion_matrix': cm.tolist(),
    }


# ==============================================================================
# Main
# ==============================================================================

def main():
    run_started = time.perf_counter()
    run_id = datetime.now().strftime('%Y%m%dT%H%M%S')

    # --- Data loading ---
    node_features, labels, label_mapping, class_weights, data_timings = (
        load_and_preprocess_data()
    )

    # --- Graph construction ---
    graph_data, graph_timings = create_graph_data(node_features, labels)

    # --- Model ---
    in_channels = graph_data.x.size(1)
    out_channels = len(label_mapping)

    model = UHGGraphSAGE(
        in_channels, HIDDEN_CHANNELS, out_channels, NUM_LAYERS, dropout=DROPOUT,
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5,
    )

    # Focal Loss with dampened class weights
    criterion = FocalLoss(alpha=class_weights.to(device), gamma=FOCAL_GAMMA)
    print(f"\nLoss: FocalLoss(gamma={FOCAL_GAMMA}, weights={WEIGHTING_STRATEGY})")

    num_params = sum(p.numel() for p in model.parameters())
    print(f"Model: {NUM_LAYERS} layers, {HIDDEN_CHANNELS} hidden, {num_params:,} params")

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
        print(f"GPU memory before training: {torch.cuda.memory_allocated()/1024**3:.2f} GB")

    # --- Training loop ---
    print(f"\nTraining (max {MAX_EPOCHS} epochs, patience={PATIENCE})...\n")

    best_val_acc = 0.0
    best_epoch = 0
    epochs_without_improvement = 0
    epoch_times = []
    train_losses = []
    val_accs = []

    for epoch in range(1, MAX_EPOCHS + 1):
        t_epoch = time.perf_counter()

        loss = train_epoch(model, graph_data, optimizer, criterion)
        val_acc = evaluate(model, graph_data, graph_data.val_mask)

        scheduler.step(val_acc)
        current_lr = optimizer.param_groups[0]['lr']

        epoch_time = time.perf_counter() - t_epoch
        epoch_times.append(epoch_time)
        train_losses.append(loss)
        val_accs.append(val_acc)

        improved = val_acc > best_val_acc
        if improved:
            best_val_acc = val_acc
            best_epoch = epoch
            epochs_without_improvement = 0
            torch.save(model.state_dict(), MODEL_SAVE_PATH)
            tag = " *"
        else:
            epochs_without_improvement += 1
            tag = ""

        if improved or epoch % 10 == 0 or epoch <= 3:
            print(f"Epoch {epoch:03d} | Loss {loss:.4f} | Val {val_acc:.4f} | "
                  f"LR {current_lr:.5f} | {epoch_time:.2f}s{tag}")

        if epochs_without_improvement >= PATIENCE:
            print(f"Early stopping at epoch {epoch}.")
            break

    train_time = sum(epoch_times)
    print(f"\nTraining complete: {len(epoch_times)} epochs, "
          f"avg {np.mean(epoch_times):.2f}s/epoch, total {train_time:.1f}s")
    print(f"Best val accuracy: {best_val_acc:.4f} (epoch {best_epoch})")

    # --- Final evaluation on test set ---
    print("\nLoading best model for final evaluation...")
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, weights_only=True))

    # UHG constraint check
    print("\n" + "=" * 80)
    print("POST-TRAINING UHG CONSTRAINT VERIFICATION")
    print("=" * 80)
    model.eval()
    with torch.no_grad():
        h = model.get_embeddings(graph_data.x, graph_data.edge_index)
        verify_uhg_constraints(h, name="after training")

    # Test evaluation (only done once, at the end)
    test_results = evaluate_detailed(
        model, graph_data, graph_data.test_mask, label_mapping, phase="Test",
    )

    print(f"\nFinal Test Accuracy: {test_results['accuracy']:.4f}")

    # --- Timing summary ---
    total_runtime = time.perf_counter() - run_started
    data_time = data_timings['total']
    graph_time = graph_timings['total']

    print("\n" + "=" * 80)
    print("TIMING SUMMARY")
    print("=" * 80)
    print(f"  Data loading:       {data_time:7.2f}s ({data_time/total_runtime*100:5.1f}%)")
    for k, v in data_timings.items():
        if k != 'total' and v > 0.01:
            print(f"    {k:20s} {v:7.2f}s")
    print(f"  Graph construction: {graph_time:7.2f}s ({graph_time/total_runtime*100:5.1f}%)")
    for k, v in graph_timings.items():
        if k != 'total' and v > 0.01:
            print(f"    {k:20s} {v:7.2f}s")
    print(f"  Training:           {train_time:7.2f}s ({train_time/total_runtime*100:5.1f}%)")
    print(f"  Total:              {total_runtime:7.2f}s ({total_runtime/60:.1f} min)")

    if torch.cuda.is_available():
        peak_mem = torch.cuda.max_memory_allocated() / 1024**3
        print(f"  Peak GPU memory:    {peak_mem:.2f} GB")

    # --- Save metrics ---
    metrics = {
        'version': 'v4.9',
        'run_id': run_id,
        'config': {
            'k_neighbors': K_NEIGHBORS,
            'hidden_channels': HIDDEN_CHANNELS,
            'num_layers': NUM_LAYERS,
            'dropout': DROPOUT,
            'focal_gamma': FOCAL_GAMMA,
            'weighting_strategy': WEIGHTING_STRATEGY,
            'sample_frac': SAMPLE_FRAC,
            'min_samples_per_class': MIN_SAMPLES_PER_CLASS,
            'pca_components': PCA_COMPONENTS,
            'lr': LR,
            'uhg_reg_lambda': UHG_REG_LAMBDA,
            'undirected': UNDIRECTED,
        },
        'results': test_results,
        'training': {
            'total_epochs': len(epoch_times),
            'best_epoch': best_epoch,
            'best_val_acc': float(best_val_acc),
            'avg_epoch_time': float(np.mean(epoch_times)),
        },
        'timing': {
            'data_load': float(data_time),
            'graph_build': float(graph_time),
            'training': float(train_time),
            'total': float(total_runtime),
        },
        'knn_backend': 'faiss' if FAISS_AVAILABLE else 'pynndescent',
    }

    metrics_file = os.path.join(RESULTS_PATH, f'metrics_v4.9_{run_id}.json')
    with open(metrics_file, 'w') as f:
        json.dump(metrics, f, indent=2)
    print(f"\nMetrics saved to: {metrics_file}")

    print("\n" + "=" * 80)
    print("UHG IDS v4.9 - Complete")
    print("=" * 80)
    print(f"  Accuracy:     {test_results['accuracy']*100:.2f}%")
    print(f"  Macro F1:     {test_results['f1_macro']:.4f}")
    print(f"  Weighted F1:  {test_results['f1_weighted']:.4f}")
    print(f"  Runtime:      {total_runtime:.0f}s ({total_runtime/60:.1f} min)")
    print(f"  KNN backend:  {'FAISS' if FAISS_AVAILABLE else 'PyNNDescent'}")
    print("=" * 80)


if __name__ == "__main__":
    main()